In [13]:
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------
# Utility
# -----------------------
def qr_orth(A):
    Q, R = np.linalg.qr(A)
    s = np.sign(np.diag(R))
    s[s == 0] = 1.0
    return Q * s

# -----------------------
# Main generator
# -----------------------
def generate_group_shared_dataset(
    out_csv: str,
    label_mode: str,        # "N" or "P"
    K: int = 30,
    N_samples: int = 500,
    dN: int = 30,
    dP: int = 10,
    sigma_N: float = 0.01,   # small common energy
    sigma_P: float = 500.0,   # large group-shared energy
    sigma_eps: float = 0.0001,
    label_noise: float = 0.3,
    seed: int = 0,
):
    """
    label_mode:
      - "N": label depends only on common N (1.2/1.3 favorable)
      - "P": label depends only on group-shared P_g (1.1 favorable)
    """

    rng = np.random.default_rng(seed)

    assert K == 30, "This setup assumes exactly 30 institutions."
    D = dN + 3 * dP  # total dimension = 60

    # ---- latent bases (orthogonal by construction via coordinates) ----
    # N: dims [0 : dN)
    # P1: [dN : dN+dP)
    # P2: [dN+dP : dN+2dP)
    # P3: [dN+2dP : dN+3dP)

    # ---- shared latent N (same for all institutions) ----
    N_lat = rng.normal(0, sigma_N, size=(N_samples, dN))

    # ---- group-shared P latents ----
    P_lat = {
        0: rng.normal(0, sigma_P, size=(N_samples, dP)),  # P1
        1: rng.normal(0, sigma_P, size=(N_samples, dP)),  # P2
        2: rng.normal(0, sigma_P, size=(N_samples, dP)),  # P3
    }

    # ---- labels ----
    if label_mode == "N":
        w = rng.normal(size=dN)
        w /= np.linalg.norm(w)
        score = N_lat @ w + rng.normal(0, label_noise, size=N_samples)
        y_global = (score > 0).astype(int)

    rows = []

    for k in range(K):
        group = k // 10  # 0,1,2 → P1,P2,P3

        # ---- observation ----
        Xk = np.zeros((N_samples, D))

        # N part (always present)
        Xk[:, :dN] = N_lat

        # group-shared P part
        start = dN + group * dP
        end = start + dP
        Xk[:, start:end] = P_lat[group]

        # small noise
        Xk += rng.normal(0, sigma_eps, size=(N_samples, D))

        # ---- labels ----
        if label_mode == "P":
            w = rng.normal(size=dP)
            w /= np.linalg.norm(w)
            score = P_lat[group] @ w + rng.normal(0, label_noise, size=N_samples)
            y = (score > 0).astype(int)
        else:
            y = y_global

        # ---- save rows ----
        for i in range(N_samples):
            row = {
                "inst_id": k,
                "sample_id": i,
                "y": int(y[i]),
            }
            for j in range(D):
                row[f"x_{j}"] = float(Xk[i, j])
            rows.append(row)

    df = pd.DataFrame(rows)
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    print(f"saved: {out_csv}")

# -----------------------
# Generate both datasets
# -----------------------
if __name__ == "__main__":
    # ① y depends on N (1.2 / 1.3 favorable)
    generate_group_shared_dataset(
        out_csv="dataset_label_N.csv",
        label_mode="N",
        seed=0,
    )

    # ② y depends on P_g (1.1 favorable)
    generate_group_shared_dataset(
        out_csv="dataset_label_P.csv",
        label_mode="P",
        seed=1,
    )


saved: dataset_label_N.csv
saved: dataset_label_P.csv


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# =========================
# Utility
# =========================
def random_orthogonal(d, rng):
    Q, _ = np.linalg.qr(rng.normal(size=(d, d)))
    return Q

# =========================
# Dataset Generator
# =========================
def generate_dataset(
    out_csv: str,
    label_mode: str,          # "N" or "P"
    K: int = 30,
    N_samples: int = 500,
    dN: int = 10,
    dP: int = 10,
    sigma_N: float = 0.3,     # low-energy common noise
    signal_P: float = 1.0,    # signal strength in P (1 dim only)
    noise_P: float = 10.0,     # huge noise in P
    sigma_eps: float = 0.2,
    label_noise: float = 0.2,
    apply_rotation: bool = False,
    seed: int = 0,
):
    """
    label_mode:
      - "N": y depends only on common N (1.2/1.3 favorable)
      - "P": y depends on 1D signal embedded in group-shared P (1.1 favorable)
    """

    rng = np.random.default_rng(seed)

    # total dimension
    D = dN + 3 * dP
    #assert D == 80

    # -------------------------
    # Latent variables
    # -------------------------
    # common N latent
    N_lat = rng.normal(0, sigma_N, size=(N_samples, dN))

    # scalar signal for P-label
    t = rng.normal(0, 1.0, size=N_samples)
    y_P = (t > 0).astype(int)

    # group-specific P latents
    P_lat = {}
    for g in range(3):
        # 1D signal direction (fixed!)
        u = np.zeros(dP)
        u[0] = 1.0

        signal = signal_P * t[:, None] * u[None, :]
        noise = noise_P * rng.normal(size=(N_samples, dP))

        P_lat[g] = signal + noise

    # -------------------------
    # N-label (shared)
    # -------------------------
    if label_mode == "N":
        wN = rng.normal(size=dN)
        wN /= np.linalg.norm(wN)
        score_N = N_lat @ wN + rng.normal(0, label_noise, size=N_samples)
        y_N = (score_N > 0).astype(int)

    # -------------------------
    # Optional global rotation
    # -------------------------
    if apply_rotation:
        R = random_orthogonal(D, rng)
    else:
        R = None

    # -------------------------
    # Build dataset
    # -------------------------
    rows = []

    for k in range(K):
        group = k // 10  # 0,1,2

        Xk = np.zeros((N_samples, D))

        # N block
        Xk[:, :dN] = N_lat

        # P_g block
        start = dN + group * dP
        end = start + dP
        Xk[:, start:end] = P_lat[group]

        # small noise everywhere
        Xk += rng.normal(0, sigma_eps, size=(N_samples, D))

        # rotation (optional)
        if R is not None:
            Xk = Xk @ R

        # labels
        if label_mode == "N":
            y = y_N
        elif label_mode == "P":
            y = y_P
        else:
            raise ValueError("label_mode must be 'N' or 'P'")

        # save
        for i in range(N_samples):
            row = {
                "inst_id": k,
                "sample_id": i,
                "y": int(y[i]),
            }
            for j in range(D):
                row[f"x_{j}"] = float(Xk[i, j])
            rows.append(row)

    df = pd.DataFrame(rows)
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    print(f"saved: {out_csv}")

# =========================
# Generate both datasets
# =========================

# ① N依存ラベル（1.2 / 1.3 が有利）
generate_dataset(
    out_csv="dataset_label_N.csv",
    label_mode="N",
    seed=0,
)

# ② P依存ラベル（1.1 が有利）
generate_dataset(
    out_csv="dataset_label_P.csv",
    label_mode="P",
    seed=1,
)


saved: dataset_label_N.csv
saved: dataset_label_P.csv


In [15]:
import numpy as np
import pandas as pd
from pathlib import Path

def random_orthogonal(d, rng):
    Q, _ = np.linalg.qr(rng.normal(size=(d, d)))
    return Q


In [16]:
def generate_dataset_v2(
    out_csv: str,
    label_mode: str,          # "N" or "P"
    K: int = 30,
    N_samples: int = 500,
    dN: int = 10,
    dP_each: int = 10,        # P1,P2,P3 それぞれの次元
    sigma_N: float = 0.3,     # N: 低エネルギー
    sigma_P: float = 3.0,     # P: 高エネルギー（大きめ推奨）
    sigma_eps: float = 0.2,   # 観測ノイズ
    label_noise: float = 0.2, # ラベルノイズ
    apply_rotation: bool = True,
    seed: int = 0,
):
    """
    設計思想：
      - 潜在空間: z = [z_N, z_P1, z_P2, z_P3]
      - 機関kは group(k) に応じて P_group だけ保持、他のPは0（観測不可）
      - 機関ごとに任意の直交変換 R_k をかけて座標依存性を消す
      - ラベル:
          N: y = 1{ ||z_N||^2 + noise > threshold }
          P: y = 1{ ||z_P_group||^2 + noise > threshold }
    """

    assert label_mode in ["N", "P"]

    rng = np.random.default_rng(seed)

    # 全次元
    dP_total = 3 * dP_each
    D = dN + dP_total

    # -------------------------
    # 共通潜在変数（全機関共通に使い回す）
    # -------------------------
    zN  = rng.normal(0, sigma_N, size=(N_samples, dN))
    zP1 = rng.normal(0, sigma_P, size=(N_samples, dP_each))
    zP2 = rng.normal(0, sigma_P, size=(N_samples, dP_each))
    zP3 = rng.normal(0, sigma_P, size=(N_samples, dP_each))

    # -------------------------
    # ラベル用の閾値（中央値でバランスよく）
    # ※ エネルギーの中央値を閾値にすると、0/1がほぼ半々になりやすい
    # -------------------------
    eN  = np.sum(zN**2, axis=1)
    eP1 = np.sum(zP1**2, axis=1)
    eP2 = np.sum(zP2**2, axis=1)
    eP3 = np.sum(zP3**2, axis=1)

    tau_N  = np.median(eN)
    tau_P1 = np.median(eP1)
    tau_P2 = np.median(eP2)
    tau_P3 = np.median(eP3)

    rows = []

    for k in range(K):
        group = k // (K // 3)  # 0,1,2（K=30なら10個ずつ）
        group = min(group, 2)

        # 機関が保持するPだけ残し、他は0（観測不能）
        if group == 0:
            zP = np.concatenate([zP1, 0*zP2, 0*zP3], axis=1)
            eP = eP1
            tau_P = tau_P1
        elif group == 1:
            zP = np.concatenate([0*zP1, zP2, 0*zP3], axis=1)
            eP = eP2
            tau_P = tau_P2
        else:
            zP = np.concatenate([0*zP1, 0*zP2, zP3], axis=1)
            eP = eP3
            tau_P = tau_P3

        z = np.concatenate([zN, zP], axis=1)  # (N_samples, D)

        # 機関ごと観測ノイズ
        Xk = z + rng.normal(0, sigma_eps, size=z.shape)

        # 機関ごとに任意の直交変換（座標系を壊す）
        if apply_rotation:
            Rk = random_orthogonal(D, rng)
            Xk = Xk @ Rk

        # ラベル生成（独立性を保証するため、NラベルはNエネルギーだけ、PラベルはPエネルギーだけ）
        if label_mode == "N":
            score = eN + rng.normal(0, label_noise, size=N_samples)
            y = (score > tau_N).astype(int)
        else:  # "P"
            score = eP + rng.normal(0, label_noise, size=N_samples)
            y = (score > tau_P).astype(int)

        # 書き出し
        for i in range(N_samples):
            row = {"inst_id": k, "sample_id": i, "y": int(y[i])}
            for j in range(D):
                row[f"x_{j}"] = float(Xk[i, j])
            rows.append(row)

    df = pd.DataFrame(rows)
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    print(f"saved: {out_csv}  shape={df.shape}  D={D}")


In [19]:
# ① N依存ラベル（Nだけで識別可能、Pだけでは原理的に無理）
generate_dataset_v2(
    out_csv="dataset_v2_label_N.csv",
    label_mode="N",
    seed=0,
    apply_rotation=False,
)

# ② P依存ラベル（その機関が保持するP部分空間だけで識別可能、Nだけでは原理的に無理）
generate_dataset_v2(
    out_csv="dataset_v2_label_P.csv",
    label_mode="P",
    seed=1,
    apply_rotation=False,
)


saved: dataset_v2_label_N.csv  shape=(15000, 43)  D=40
saved: dataset_v2_label_P.csv  shape=(15000, 43)  D=40


In [18]:
dfN = pd.read_csv("dataset_v2_label_N.csv")
dfP = pd.read_csv("dataset_v2_label_P.csv")

print("N-label y mean:", dfN["y"].mean())
print("P-label y mean:", dfP["y"].mean())

print("\nPer-inst label mean (first 5) N:")
print(dfN.groupby("inst_id")["y"].mean().head())

print("\nPer-inst label mean (first 5) P:")
print(dfP.groupby("inst_id")["y"].mean().head())


N-label y mean: 0.5144
P-label y mean: 0.5000666666666667

Per-inst label mean (first 5) N:
inst_id
0    0.520
1    0.536
2    0.492
3    0.512
4    0.536
Name: y, dtype: float64

Per-inst label mean (first 5) P:
inst_id
0    0.500
1    0.500
2    0.500
3    0.500
4    0.502
Name: y, dtype: float64


In [28]:
import numpy as np
import pandas as pd
from pathlib import Path

# =========================
# Utility
# =========================
def random_orthogonal(d, rng):
    Q, _ = np.linalg.qr(rng.normal(size=(d, d)))
    return Q

# =========================
# Dataset Generator
# =========================
def generate_factor_dataset(
    out_csv: str,
    label_mode: str,          # "N" or "P"
    K: int = 30,              # number of institutions
    N_samples: int = 500,
    d_s: int = 10,            # common factor dimension
    d_t: int = 10,            # per-type private factor dimension
    sigma_s: float = 2,     # common factor scale (low energy)
    sigma_t: float = 200.0,     # private factor scale (high energy)
    sigma_eps: float = 0.02,  # observation noise (small)
    label_noise: float = 0.2,
    seed: int = 0,
):
    """
    label_mode:
      - "N": label depends on common factor s
      - "P": label depends on private factor t_k
    """

    assert label_mode in ["N", "P"]
    rng = np.random.default_rng(seed)

    # -------------------------
    # Dimensions
    # -------------------------
    D = d_s + 3 * d_t

    # -------------------------
    # Orthonormal bases (fixed)
    # -------------------------
    Q = random_orthogonal(D, rng)
    B_S  = Q[:, :d_s]
    B_T0 = Q[:, d_s : d_s + d_t]
    B_T1 = Q[:, d_s + d_t : d_s + 2 * d_t]
    B_T2 = Q[:, d_s + 2 * d_t : d_s + 3 * d_t]

    # -------------------------
    # Shared common factors
    # -------------------------
    s = rng.normal(0, sigma_s, size=(N_samples, d_s))

    # label weights
    w_s = rng.normal(size=d_s)
    w_s /= np.linalg.norm(w_s)

    w_t = rng.normal(size=d_t)
    w_t /= np.linalg.norm(w_t)

    rows = []

    for k in range(K):
        group = k // (K // 3)   # 0,1,2
        group = min(group, 2)

        # select private basis
        if group == 0:
            B_T = B_T0
        elif group == 1:
            B_T = B_T1
        else:
            B_T = B_T2

        # private factors for this institution
        t = rng.normal(0, sigma_t, size=(N_samples, d_t))

        # observed data
        X = (
            s @ B_S.T
            + t @ B_T.T
            + rng.normal(0, sigma_eps, size=(N_samples, D))
        )

        # labels
        if label_mode == "N":
            score = s @ w_s + rng.normal(0, label_noise, size=N_samples)
        else:  # "P"
            score = t @ w_t + rng.normal(0, label_noise, size=N_samples)

        y = (score > 0).astype(int)

        # save rows
        for i in range(N_samples):
            row = {
                "inst_id": k,
                "sample_id": i,
                "y": int(y[i]),
            }
            for j in range(D):
                row[f"x_{j}"] = float(X[i, j])
            rows.append(row)

    df = pd.DataFrame(rows)
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)
    print(f"saved: {out_csv}  shape={df.shape}  D={D}")


In [27]:
# 共通因子ラベル（Nラベル）
generate_factor_dataset(
    out_csv="dataset_label_N.csv",
    label_mode="N",
    seed=0,
)

# 個別因子ラベル（Pラベル）
generate_factor_dataset(
    out_csv="dataset_label_P.csv",
    label_mode="P",
    seed=1,
)


saved: dataset_label_N.csv  shape=(30000, 43)  D=40
saved: dataset_label_P.csv  shape=(30000, 43)  D=40
